<a href="https://colab.research.google.com/github/alexandergribenchenko/Data_Science_Finanzas/blob/main/Binance_P2P.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from datetime import datetime


In [5]:
import requests
import pandas as pd

def obtener_anuncios_p2p(fiat="COP", asset="USDT", trade_type="BUY", pagina=1, filas=20):
    """
    Consulta el endpoint público que usa la web de Binance P2P
    para listar anuncios de compra/venta.

    Parametros:
        fiat: moneda fiat, ej. "COP", "VES", "ARS", "MXN"
        asset: criptomoneda, ej. "USDT", "BTC"
        trade_type: "BUY" o "SELL" (desde la perspectiva del que consulta)
        pagina: numero de pagina de resultados
        filas: cantidad de anuncios por pagina (max 20 recomendado)
    """
    url = "https://p2p.binance.com/bapi/c2c/v2/friendly/c2c/adv/search"

    payload = {
        "fiat": fiat,
        "page": pagina,
        "rows": filas,
        "tradeType": trade_type,
        "asset": asset,
        "countries": [],
        "proMerchantAds": False,
        "shieldMerchantAds": False,
        "publisherType": None,
        "payTypes": [],
        "classifies": ["mass", "profession", "fiat_trade"]
    }

    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    respuesta = requests.post(url, json=payload, headers=headers, timeout=10)
    respuesta.raise_for_status()
    datos = respuesta.json()

    anuncios = datos.get("data", [])
    filas_tabla = []

    for anuncio in anuncios:
        adv = anuncio.get("adv", {})
        advertiser = anuncio.get("advertiser", {})
        metodos_pago = [m.get("tradeMethodName") for m in adv.get("tradeMethods", [])]

        filas_tabla.append({
            "comerciante": advertiser.get("nickName"),
            "precio": float(adv.get("price", 0)),
            "moneda": fiat,
            "disponible": float(adv.get("surplusAmount", 0)),
            "limite_min": float(adv.get("minSingleTransAmount", 0)),
            "limite_max": float(adv.get("dynamicMaxSingleTransAmount", 0)),
            "metodos_pago": ", ".join(metodos_pago),
            "ordenes_completadas": advertiser.get("monthOrderCount"),
            "tasa_finalizacion_%": round(float(advertiser.get("monthFinishRate", 0)) * 100, 2),
            "insignia_verificado": advertiser.get("userType"),
            "tiempo_respuesta_min": adv.get("payTimeLimit", 0) // 60 if adv.get("payTimeLimit") else None,
        })

    return pd.DataFrame(filas_tabla)


# Llamada y visualización directa del DataFrame
df_vendedores = obtener_anuncios_p2p(fiat="COP", asset="USDT", trade_type="BUY", filas=20)
df_vendedores

,comerciante,precio,moneda,disponible,limite_min,limite_max,metodos_pago,ordenes_completadas,tasa_finalizacion_%,insignia_verificado,tiempo_respuesta_min
0,JimenaR-Coins,3499.35,COP,40052.49,40000.0,25000000.0,"Bancolombia S.A, Nequi, Bre-B Keys",7991,99.9,merchant,0
1,Cambios_Avg,3329.90,COP,149.67,300000.0,497143.0,Bancolombia S.A,141,96.6,user,0
2,Jada_store,3330.00,COP,300.00,100000.0,996508.0,Bancolombia S.A,137,93.9,user,0
3,CF2-Exchanges,3330.00,COP,1000.00,3000000.0,3000001.0,"Bancolombia S.A, Nequi",2169,99.6,merchant,0
4,GlobalTraderVA,3331.99,COP,1535.00,500000.0,4000000.0,Bancolombia S.A,17,100.0,user,0
5,cripto86,3332.00,COP,36.44,100000.0,121115.0,Bancolombia S.A,91,96.9,user,0
6,MoneyExpress1,3332.00,COP,500.00,500000.0,1661845.0,Bancolombia S.A,568,96.5,user,0
7,PAGO-EXPREXX,3334.79,COP,186.74,596000.0,621650.0,Bancolombia S.A,2698,100.0,merchant,0
8,User-j6598,3335.00,COP,10000.00,15000000.0,25000000.0,"Bancolombia S.A, Bre-B Keys",70,100.0,user,0
9,Yorbe_Studio,3335.00,COP,840.57,1000000.0,2796310.0,Bancolombia S.A,162,95.9,user,0


In [7]:
import requests
import pandas as pd

def obtener_anuncios_p2p(fiat="COP", asset="USDT", trade_type="BUY", pagina=1, filas=20):
    url = "https://p2p.binance.com/bapi/c2c/v2/friendly/c2c/adv/search"

    payload = {
        "fiat": fiat,
        "page": pagina,
        "rows": filas,
        "tradeType": trade_type,
        "asset": asset,
        "countries": [],
        "proMerchantAds": False,
        "shieldMerchantAds": False,
        "publisherType": None,
        "payTypes": [],
        "classifies": ["mass", "profession", "fiat_trade"]
    }

    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    respuesta = requests.post(url, json=payload, headers=headers, timeout=10)
    respuesta.raise_for_status()
    datos = respuesta.json()

    anuncios = datos.get("data", [])
    filas_tabla = []

    for anuncio in anuncios:
        adv = anuncio.get("adv", {})
        advertiser = anuncio.get("advertiser", {})
        metodos_pago = [m.get("tradeMethodName") for m in adv.get("tradeMethods", [])]

        filas_tabla.append({
            "codigo_vendedor": advertiser.get("userNo"),   # <-- el advertiserNo de la URL
            "comerciante": advertiser.get("nickName"),
            "precio": float(adv.get("price", 0)),
            "moneda": fiat,
            "disponible": float(adv.get("surplusAmount", 0)),
            "limite_min": float(adv.get("minSingleTransAmount", 0)),
            "limite_max": float(adv.get("dynamicMaxSingleTransAmount", 0)),
            "metodos_pago": ", ".join(metodos_pago),
            "ordenes_completadas": advertiser.get("monthOrderCount"),
            "tasa_finalizacion_%": round(float(advertiser.get("monthFinishRate", 0)) * 100, 2),
            "insignia_verificado": advertiser.get("userType"),
            "tiempo_respuesta_min": adv.get("payTimeLimit", 0) // 60 if adv.get("payTimeLimit") else None,
        })

    return pd.DataFrame(filas_tabla)


df_vendedores = obtener_anuncios_p2p(fiat="COP", asset="USDT", trade_type="BUY", filas=20)
df_vendedores

,codigo_vendedor,comerciante,precio,moneda,disponible,limite_min,limite_max,metodos_pago,ordenes_completadas,tasa_finalizacion_%,insignia_verificado,tiempo_respuesta_min
0,sa19c588fa9a730f8a717a82950138f76,JimenaR-Coins,3499.35,COP,38755.22,40000.0,25000000.0,"Bancolombia S.A, Nequi, Bre-B Keys",7998,99.9,merchant,0
1,s427c4cfcdd8139008b7c22e159fec82c,Jada_store,3330.00,COP,88.23,100000.0,293073.0,Bancolombia S.A,137,93.9,user,0
2,s22b52acaaf7239eba8b4e4976f0bcd48,inversiones_10,3330.00,COP,448.26,200000.0,1488983.0,"Bancolombia S.A, Bre-B Keys",75,100.0,user,0
3,sa9af913c06d63db793b71653c065d2a7,Kadaki_fast,3330.00,COP,10020.00,10999312.0,11000000.0,Bancolombia S.A,95,84.1,merchant,0
4,s60b30e33ad7e3e519f852c0476dd6dc3,cripto86,3332.00,COP,36.44,100000.0,121115.0,Bancolombia S.A,91,96.9,user,0
5,s7d844fbde00837f390d5ce5a9cbf3848,PAGO-EXPREXX,3334.79,COP,125.31,416000.0,417152.0,Bancolombia S.A,2699,100.0,merchant,0
6,s6a69305bd21936aab730876bd12e2829,Fast Cash,3334.98,COP,240.50,800000.0,800461.0,"Bancolombia S.A, Bre-B Keys",164,94.3,merchant,0
7,sd56376eac3913755bee8ac348b5c4b94,User-j6598,3335.00,COP,10000.00,15000000.0,25000000.0,"Bancolombia S.A, Bre-B Keys",70,100.0,user,0
8,sb8e5bb52d8543319a18fe6d36d80945e,Yorbe_Studio,3335.00,COP,840.57,1000000.0,2796310.0,Bancolombia S.A,162,95.9,user,0
9,s2b8615acdb9b335790d526be3b558721,capiramirez,3335.00,COP,194.20,630000.0,640000.0,"Bancolombia S.A, Nequi",12,100.0,user,0


In [6]:
import requests
import pandas as pd
import re
import json

def obtener_perfil_vendedor(advertiser_no):
    """
    Extrae la informacion del perfil de un vendedor P2P de Binance
    a partir de su pagina publica de detalle.
    """
    url = f"https://c2c.binance.com/en/advertiserDetail?advertiserNo={advertiser_no}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    respuesta = requests.get(url, headers=headers, timeout=10)
    respuesta.raise_for_status()
    html = respuesta.text

    match = re.search(
        r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>',
        html, re.DOTALL
    )
    if not match:
        raise ValueError("No se encontro __NEXT_DATA__. Puede necesitar Selenium/Playwright.")

    return json.loads(match.group(1))


# Paso 1: correlo así primero para ver la estructura real
advertiser_no = "sa19c588fa9a730f8a717a82950138f76"
datos = obtener_perfil_vendedor(advertiser_no)

# Esto te va a mostrar cómo está organizado el JSON por dentro
print(list(datos["props"]["pageProps"].keys()))

ValueError: No se encontro __NEXT_DATA__. Puede necesitar Selenium/Playwright.